# Parsex High-Performance "Data Engine" Notebook

This notebook demonstrates the Phase 2 HPC optimization: **Direct Python Memory Binding**.
By bypassing disk I/O (JSON parsing) and exposing the C++ Structure-of-Arrays (SoA) engine directly to Python via `pybind11`, we achieve unprecedented ML data generation throughput.

We will dynamically simulate 1,000,000 variations of a Voltage Divider and train a PyTorch surrogate Neural Network inside this same notebook loop.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../build")) # Link the compiled parsex.so library

import parsex
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time
from torch.utils.data import TensorDataset, DataLoader

print(f"CUDA Available for PyTorch: {torch.cuda.is_available()}")

### 1. Load the Base Topology Blueprint

In [ ]:
# Load the circuit topology ONCE. (0 string operations after this point)
topo = parsex.CircuitTopology()
if not topo.load_from_json("../circuits/voltage_divider.json"):
    raise Exception("Failed to load topology")

print(f"Topology loaded:")
print(f"  Nodes: {topo.num_nodes}")
print(f"  Resistors: {topo.num_resistors}")
print(f"  Voltage Sources: {topo.num_voltage_sources}")

### 2. Generate 1,000,000 Parameter Variations in Memory
We simulate an RL agent or Generative Model exploring 1M different parameters.

In [ ]:
BATCH_SIZE = 1_000_000
params = parsex.ParameterBatch(BATCH_SIZE, topo.num_resistors, topo.num_voltage_sources, topo.num_current_sources)

print(f"Allocating memory for {BATCH_SIZE:,} circuits...")

# Ground truth distributions for the ML dataset
r_values = np.random.uniform(500.0, 5000.0, size=(topo.num_resistors * BATCH_SIZE)).astype(np.float64)
v_values = np.random.uniform(1.0, 12.0, size=(topo.num_voltage_sources * BATCH_SIZE)).astype(np.float64)
i_values = np.array([], dtype=np.float64) # No current sources

# Zero-copy binding into the C++ SoA engine
params.set_r_values(r_values)
params.set_v_values(v_values)

### 3. Execute Parsex HPC Engine
OpenMP Batched SIMD Gaussian Elimination directly on the Memory Block.

In [ ]:
print("Starting Parsex C++ Backend...")
start_time = time.time()

# Solves the MNA matrix for all 1,000,000 circuits simultaneously
voltages = parsex.solve_batch(topo, params) 

end_time = time.time()
cpp_time = end_time - start_time
throughput = BATCH_SIZE / cpp_time
print(f"\n--- HPC Simulation Complete ---")
print(f"Time taken: {cpp_time:.4f} seconds")
print(f"Throughput: {throughput:,.2f} circuits / sec")
print(f"Voltage Array Shape: {voltages.shape}")

### 4. Train a PyTorch Surrogate Model
We now have a massive Ground Truth dataset (`X` = parameters, `Y` = voltages). Let's train an MLP to predict simulator outputs instantaneously.

In [ ]:
# Prepare inputs X (R1, R2, V1)
r_reshaped = r_values.reshape(topo.num_resistors, BATCH_SIZE).T # Shape: (Batch, num_R)
v_reshaped = v_values.reshape(topo.num_voltage_sources, BATCH_SIZE).T # Shape: (Batch, num_V)
X_np = np.hstack([r_reshaped, v_reshaped])
Y_np = voltages

# Normalization
X_mean, X_std = X_np.mean(axis=0), X_np.std(axis=0)
Y_mean, Y_std = Y_np.mean(axis=0), Y_np.std(axis=0)
X_norm = (X_np - X_mean) / X_std
Y_norm = (Y_np - Y_mean) / Y_std

X_tensor = torch.tensor(X_norm, dtype=torch.float32)
Y_tensor = torch.tensor(Y_norm, dtype=torch.float32)

dataset = TensorDataset(X_tensor, Y_tensor)
dataloader = DataLoader(dataset, batch_size=4096, shuffle=True)

class SurrogateModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
        
    def forward(self, x):
        return self.net(x)

model = SurrogateModel(input_dim=X_np.shape[1], output_dim=Y_np.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Training Surrogate Model...")
epochs = 3
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_Y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_Y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {total_loss/len(dataloader):.6f}")
print("Training Complete!")

### 5. Prediction Verification

In [ ]:
sample_idx = 42
with torch.no_grad():
    pred_norm = model(X_tensor[sample_idx])

pred_v = (pred_norm.numpy() * Y_std) + Y_mean
actual_v = Y_np[sample_idx]

print(f"Parameters (R1, R2, V1): {X_np[sample_idx]}")
print("\n--- Node Voltages ---")
for i, v in enumerate(pred_v):
    # Skip node 0 (ground) if it's implicitly handled, but map it back if needed.
    # varsToReturn mapping from bindings.cpp is roughly nodes > 0
    print(f"Node {i+1}: Parsex={actual_v[i]:.4f}V | Surr. AI={v:.4f}V | Error={abs(actual_v[i]-v):.4f}V")